# Figure 3 — Spatial distribution of groundwater data and the target variable (Depth to Water)

Regenerates **Figure 3** of the RIFT manuscript with four panels:

- **(A)** Basin-aggregated records — one bubble per basin (coloured by mean DTW, sized by record count).
- **(B)** Individual records — every cleaned observation, coloured by DTW.
- **(C)** DTW distribution (mean / median).
- **(D)** Log-transformed distribution, `log(1 + DTW)`.

The **outline of Chile** (Natural Earth 50 m, simplified) is embedded directly in this notebook, so the
maps draw the country boundary with **no shapefile, geopandas, cartopy, or internet** required.
Reads the same Excel file and columns as the analysis, applies the same cleaning (drop missing
coordinates/DTW; remove negative depths), and exports **PNG (400 dpi), PDF, and SVG**.


In [ ]:
# === Configuration =========================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import MaxNLocator

DATA_PATH  = r"C:\Users\m20000907\OneDrive - Colorado School of Mines\projects_2025\chile_groundwater\results\chile_2025\groundwater_chile_and_elevation_dataset_2025_with_GEE.xlsx"
OUTPUT_DIR = r"C:\Users\m20000907\OneDrive - Colorado School of Mines\projects_2025\chile_groundwater\draft_for_AI_model\figures_7_26"

LON_COL   = "Longitude_GCS_WGS_1984"
LAT_COL   = "Latitude_GCS_WGS_1984"
DTW_COL   = "Depth to water (m)"
BASIN_COL = "Basin"

TEST_MODE = False   # True -> synthetic data, no Excel needed

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Output folder:", OUTPUT_DIR)


In [ ]:
# === Embedded Chile boundary (Natural Earth 50 m, Douglas-Peucker simplified) ===
# Two rings: mainland + Chiloe. Each is a list of (lon, lat) vertices.
CHILE_OUTLINE = [
  [(-70.418, -18.346), (-70.184, -18.325), (-69.926, -18.206), (-69.803, -17.99), (-69.852, -17.704), (-69.806, -17.665), (-69.685, -17.65), (-69.511, -17.506), (-69.495, -17.62), (-69.358, -17.772), (-69.313, -17.943), (-69.094, -18.05), (-69.145, -18.144), (-69.092, -18.282), (-68.968, -18.968), (-68.463, -19.433), (-68.698, -19.721), (-68.578, -19.857), (-68.561, -19.967), (-68.6, -20.045), (-68.756, -20.091), (-68.689, -20.31), (-68.759, -20.378), (-68.761, -20.416), (-68.745, -20.459), (-68.484, -20.628), (-68.563, -20.72), (-68.558, -20.902), (-68.435, -20.948), (-68.197, -21.3), (-68.186, -21.619), (-68.112, -21.753), (-68.077, -21.983), (-67.988, -22.057), (-67.95, -22.334), (-67.882, -22.493), (-67.879, -22.823), (-67.794, -22.879), (-67.58, -22.892), (-67.195, -22.822), (-67.009, -23.001), (-67.356, -24.034), (-68.25, -24.392), (-68.507, -24.63), (-68.562, -24.747), (-68.562, -24.838), (-68.527, -24.899), (-68.466, -24.925), (-68.428, -25.051), (-68.384, -25.092), (-68.431, -25.149), (-68.496, -25.163), (-68.592, -25.42), (-68.6, -25.486), (-68.415, -26.154), (-68.576, -26.352), (-68.592, -26.47), (-68.485, -26.67), (-68.319, -26.878), (-68.319, -26.973), (-68.346, -27.028), (-68.537, -27.085), (-68.592, -27.14), (-68.652, -27.148), (-68.71, -27.104), (-68.846, -27.154), (-68.942, -27.405), (-68.999, -27.449), (-69.174, -27.925), (-69.251, -27.974), (-69.437, -28.193), (-69.489, -28.201), (-69.527, -28.286), (-69.657, -28.414), (-69.735, -28.641), (-69.743, -28.784), (-69.828, -29.103), (-69.996, -29.25), (-70.027, -29.324), (-69.928, -29.769), (-69.96, -30.078), (-69.863, -30.12), (-69.844, -30.175), (-69.956, -30.358), (-70.102, -30.388), (-70.153, -30.361), (-70.348, -30.902), (-70.309, -31.023), (-70.388, -31.121), (-70.473, -31.113), (-70.52, -31.148), (-70.585, -31.569), (-70.45, -31.842), (-70.394, -31.884), (-70.331, -31.881), (-70.254, -31.958), (-70.291, -32.031), (-70.356, -32.042), (-70.364, -32.083), (-70.32, -32.267), (-70.258, -32.31), (-70.23, -32.431), (-70.17, -32.472), (-70.177, -32.626), (-70.116, -32.807), (-70.022, -32.885), (-70.093, -33.027), (-70.085, -33.202), (-70.02, -33.271), (-69.896, -33.251), (-69.82, -33.284), (-69.798, -33.399), (-69.883, -33.601), (-69.894, -33.731), (-69.852, -34.224), (-70.052, -34.301), (-70.101, -34.432), (-70.29, -34.733), (-70.393, -35.147), (-70.555, -35.247), (-70.532, -35.308), (-70.47, -35.326), (-70.457, -35.452), (-70.416, -35.523), (-70.38, -35.772), (-70.416, -35.879), (-70.405, -36.062), (-70.457, -36.133), (-70.563, -36.146), (-70.722, -36.283), (-70.749, -36.393), (-70.905, -36.42), (-71.056, -36.524), (-71.066, -36.644), (-71.192, -36.844), (-71.118, -37.114), (-71.2, -37.3), (-71.135, -37.445), (-71.187, -37.631), (-71.168, -37.762), (-71.028, -38.041), (-70.968, -38.446), (-70.848, -38.542), (-70.897, -38.681), (-70.952, -38.738), (-71.087, -38.758), (-71.286, -38.845), (-71.402, -38.935), (-71.426, -38.986), (-71.42, -39.287), (-71.539, -39.602), (-71.654, -39.594), (-71.72, -39.635), (-71.638, -39.887), (-71.66, -40.021), (-71.704, -40.095), (-71.802, -40.125), (-71.818, -40.177), (-71.801, -40.244), (-71.695, -40.335), (-71.709, -40.382), (-71.805, -40.439), (-71.932, -40.692), (-71.941, -40.789), (-71.873, -40.893), (-71.892, -41.393), (-71.871, -41.561), (-71.911, -41.65), (-71.75, -42.047), (-71.761, -42.101), (-71.861, -42.148), (-72.026, -42.148), (-72.108, -42.252), (-72.125, -42.298), (-72.078, -42.358), (-72.053, -42.473), (-72.144, -42.577), (-72.114, -42.777), (-72.146, -42.99), (-72.102, -43.066), (-72.055, -43.102), (-71.781, -43.167), (-71.751, -43.237), (-71.764, -43.295), (-71.905, -43.348), (-71.905, -43.44), (-71.751, -43.59), (-71.737, -43.705), (-71.795, -43.753), (-71.68, -43.93), (-71.812, -44.106), (-71.82, -44.383), (-71.213, -44.441), (-71.151, -44.494), (-71.16, -44.56), (-71.221, -44.631), (-71.261, -44.763), (-71.358, -44.785), (-71.455, -44.75), (-71.957, -44.792), (-72.064, -44.772), (-72.042, -44.904), (-71.596, -44.979), (-71.443, -45.168), (-71.354, -45.23), (-71.349, -45.332), (-71.49, -45.438), (-71.508, -45.513), (-71.693, -45.535), (-71.746, -45.579), (-71.773, -45.724), (-71.751, -45.839), (-71.68, -45.879), (-71.632, -45.954), (-71.684, -46.042), (-71.876, -46.161), (-71.762, -46.32), (-71.7, -46.651), (-71.733, -46.706), (-71.94, -46.831), (-71.963, -47.016), (-71.954, -47.087), (-71.901, -47.144), (-71.905, -47.202), (-72.042, -47.241), (-72.103, -47.343), (-72.346, -47.493), (-72.342, -47.572), (-72.518, -47.876), (-72.509, -47.973), (-72.408, -48.016), (-72.328, -48.11), (-72.293, -48.229), (-72.355, -48.366), (-72.498, -48.417), (-72.608, -48.519), (-72.586, -48.663), (-72.614, -48.793), (-72.728, -48.896), (-73.034, -49.014), (-73.149, -49.188), (-73.135, -49.301), (-73.462, -49.314), (-73.484, -49.398), (-73.554, -49.464), (-73.576, -49.583), (-73.47, -49.795), (-73.529, -49.911), (-73.501, -50.125), (-73.387, -50.231), (-73.312, -50.362), (-73.252, -50.558), (-73.153, -50.738), (-73.082, -50.76), (-72.866, -50.653), (-72.62, -50.648), (-72.51, -50.608), (-72.393, -50.634), (-72.34, -50.682), (-72.276, -50.91), (-72.307, -51.033), (-72.377, -51.095), (-72.359, -51.17), (-72.302, -51.223), (-72.303, -51.299), (-72.408, -51.541), (-72.269, -51.691), (-72.137, -51.744), (-71.953, -51.88), (-71.971, -51.964), (-71.919, -51.99), (-69.96, -52.008), (-69.488, -52.136), (-69.206, -52.136), (-68.461, -52.29), (-68.443, -52.357), (-69.241, -52.205), (-69.447, -52.269), (-69.62, -52.465), (-69.907, -52.514), (-70.391, -52.661), (-70.563, -52.673), (-70.795, -52.769), (-70.839, -52.89), (-70.821, -52.963), (-70.952, -53.227), (-70.984, -53.374), (-70.948, -53.57), (-70.996, -53.779), (-71.298, -53.883), (-71.694, -53.803), (-72.174, -53.632), (-72.377, -53.471), (-72.413, -53.35), (-72.306, -53.254), (-71.942, -53.234), (-71.853, -53.286), (-71.828, -53.398), (-71.903, -53.496), (-71.892, -53.524), (-71.791, -53.485), (-71.741, -53.233), (-71.4, -53.107), (-71.289, -53.034), (-71.163, -52.888), (-71.155, -52.846), (-71.227, -52.811), (-71.388, -52.764), (-71.898, -53.002), (-72.278, -53.132), (-72.493, -53.291), (-72.549, -53.461), (-72.727, -53.42), (-73.053, -53.243), (-72.916, -53.122), (-72.889, -52.872), (-72.728, -52.762), (-72.676, -52.749), (-72.632, -52.774), (-72.627, -52.818), (-72.453, -52.814), (-72.118, -52.65), (-71.797, -52.683), (-71.591, -52.661), (-71.511, -52.605), (-71.812, -52.537), (-72.226, -52.521), (-72.315, -52.539), (-72.438, -52.626), (-72.504, -52.56), (-72.712, -52.536), (-72.777, -52.577), (-72.766, -52.643), (-72.802, -52.712), (-72.932, -52.782), (-73.02, -52.892), (-73.023, -53.022), (-73.122, -53.074), (-73.338, -53.055), (-73.508, -52.904), (-73.645, -52.837), (-73.241, -52.707), (-73.073, -52.535), (-73.124, -52.488), (-73.184, -52.488), (-73.178, -52.563), (-73.244, -52.624), (-73.382, -52.595), (-73.586, -52.686), (-73.711, -52.662), (-73.915, -52.688), (-74.014, -52.639), (-74.036, -52.577), (-74.0, -52.513), (-74.037, -52.403), (-74.151, -52.383), (-74.296, -52.118), (-74.195, -52.12), (-73.834, -52.234), (-73.703, -52.199), (-73.684, -52.078), (-73.532, -52.153), (-73.26, -52.158), (-72.944, -52.047), (-72.795, -51.95), (-72.695, -51.985), (-72.695, -52.045), (-72.588, -52.145), (-72.571, -52.2), (-72.583, -52.254), (-72.714, -52.357), (-72.677, -52.385), (-72.631, -52.372), (-72.569, -52.334), (-72.523, -52.255), (-72.524, -52.17), (-72.625, -52.007), (-72.625, -51.946), (-72.494, -51.848), (-72.49, -51.764), (-72.543, -51.706), (-72.761, -51.573), (-73.127, -51.44), (-73.169, -51.454), (-73.197, -51.478), (-72.789, -51.614), (-72.649, -51.695), (-72.583, -51.737), (-72.6, -51.799), (-72.928, -51.86), (-73.383, -52.07), (-73.518, -52.041), (-73.65, -51.856), (-73.753, -51.796), (-73.858, -51.79), (-73.894, -51.758), (-73.973, -51.784), (-74.197, -51.681), (-74.07, -51.579), (-73.93, -51.618), (-73.896, -51.331), (-73.94, -51.266), (-74.121, -51.195), (-74.332, -51.195), (-74.815, -51.063), (-75.055, -50.786), (-75.095, -50.681), (-74.686, -50.662), (-74.649, -50.618), (-74.776, -50.47), (-74.644, -50.361), (-74.366, -50.488), (-74.139, -50.818), (-73.807, -50.938), (-73.825, -50.836), (-73.741, -50.697), (-73.618, -50.651), (-73.693, -50.57), (-73.654, -50.493), (-73.75, -50.54), (-73.892, -50.783), (-73.978, -50.827), (-74.197, -50.61), (-74.186, -50.485), (-73.95, -50.511), (-74.425, -50.35), (-74.63, -50.194), (-74.334, -49.975), (-74.019, -50.023), (-73.959, -49.995), (-74.011, -49.929), (-74.073, -49.949), (-74.171, -49.907), (-74.324, -49.783), (-74.291, -49.604), (-74.102, -49.555), (-73.892, -49.624), (-73.836, -49.609), (-73.892, -49.523), (-74.094, -49.43), (-74.023, -49.244), (-74.015, -49.091), (-73.935, -49.021), (-74.028, -49.026), (-74.221, -49.501), (-74.367, -49.4), (-74.382, -48.794), (-74.341, -48.596), (-74.176, -48.494), (-74.057, -48.504), (-74.009, -48.475), (-74.172, -48.427), (-74.343, -48.493), (-74.474, -48.464), (-74.499, -48.362), (-74.577, -48.274), (-74.585, -47.999), (-74.25, -48.045), (-73.854, -48.042), (-73.528, -48.198), (-73.384, -48.177), (-73.391, -48.146), (-73.501, -48.107), (-73.61, -47.994), (-73.716, -47.655), (-73.748, -47.661), (-73.847, -47.867), (-73.941, -47.929), (-74.227, -47.969), (-74.351, -47.944), (-74.376, -47.83), (-74.609, -47.758), (-74.655, -47.702), (-74.534, -47.568), (-74.243, -47.679), (-74.134, -47.591), (-74.324, -47.531), (-74.483, -47.43), (-74.403, -47.328), (-74.158, -47.183), (-74.208, -47.083), (-74.152, -46.974), (-74.314, -46.788), (-74.454, -46.767), (-74.484, -46.795), (-74.467, -46.864), (-74.512, -46.885), (-74.691, -46.864), (-75.006, -46.741), (-75.053, -46.628), (-74.984, -46.512), (-75.146, -46.6), (-75.478, -46.662), (-75.54, -46.699), (-75.565, -46.729), (-75.446, -46.751), (-75.386, -46.863), (-75.43, -46.935), (-75.497, -46.94), (-75.635, -46.863), (-75.708, -46.775), (-75.706, -46.705), (-75.657, -46.61), (-74.924, -46.16), (-75.075, -46.004), (-75.067, -45.875), (-74.763, -45.824), (-74.463, -45.841), (-74.158, -45.767), (-74.082, -45.678), (-74.123, -45.496), (-74.038, -45.418), (-73.957, -45.404), (-73.825, -45.447), (-73.96, -45.835), (-74.061, -45.947), (-74.02, -46.056), (-74.082, -46.132), (-74.393, -46.217), (-74.372, -46.246), (-74.09, -46.222), (-73.968, -46.154), (-73.879, -45.847), (-73.735, -45.812), (-73.695, -45.86), (-73.708, -46.07), (-73.811, -46.377), (-73.949, -46.533), (-73.944, -46.572), (-73.845, -46.566), (-73.77, -46.5), (-73.662, -46.297), (-73.629, -45.987), (-73.592, -45.899), (-73.594, -45.777), (-73.757, -45.703), (-73.78, -45.628), (-73.731, -45.48), (-73.55, -45.484), (-73.266, -45.346), (-72.934, -45.452), (-72.976, -45.393), (-73.226, -45.255), (-73.445, -45.238), (-73.405, -45.102), (-73.362, -44.978), (-73.078, -44.92), (-72.739, -44.734), (-72.68, -44.594), (-72.664, -44.436), (-72.828, -44.395), (-73.265, -44.169), (-73.224, -43.898), (-73.069, -43.862), (-72.997, -43.632), (-73.101, -43.455), (-73.076, -43.324), (-72.94, -43.211), (-72.878, -43.048), (-72.758, -43.039), (-72.766, -42.908), (-72.845, -42.808), (-72.848, -42.669), (-72.774, -42.505), (-72.632, -42.51), (-72.785, -42.301), (-72.773, -42.258), (-72.631, -42.2), (-72.548, -42.256), (-72.43, -42.434), (-72.412, -42.388), (-72.499, -41.981), (-72.624, -42.011), (-72.738, -41.995), (-72.824, -41.909), (-72.66, -41.742), (-72.486, -41.722), (-72.36, -41.649), (-72.318, -41.499), (-72.359, -41.514), (-72.428, -41.646), (-72.542, -41.691), (-72.67, -41.659), (-72.805, -41.544), (-72.953, -41.515), (-73.015, -41.544), (-73.174, -41.747), (-73.242, -41.781), (-73.521, -41.797), (-73.735, -41.742), (-73.624, -41.581), (-73.711, -41.574), (-73.811, -41.517), (-73.855, -41.446), (-73.876, -41.319), (-73.966, -41.118), (-73.984, -40.974), (-73.92, -40.872), (-73.784, -40.468), (-73.669, -40.082), (-73.671, -39.963), (-73.41, -39.789), (-73.25, -39.422), (-73.226, -39.224), (-73.52, -38.509), (-73.533, -38.367), (-73.465, -38.04), (-73.662, -37.699), (-73.665, -37.59), (-73.603, -37.479), (-73.662, -37.341), (-73.602, -37.188), (-73.375, -37.224), (-73.216, -37.167), (-73.118, -36.688), (-73.007, -36.643), (-72.875, -36.39), (-72.778, -35.979), (-72.587, -35.76), (-72.624, -35.586), (-72.387, -35.24), (-72.224, -35.096), (-72.056, -34.616), (-71.992, -34.288), (-72.003, -34.165), (-71.831, -33.82), (-71.664, -33.653), (-71.636, -33.519), (-71.696, -33.429), (-71.743, -33.095), (-71.592, -32.97), (-71.452, -32.66), (-71.461, -32.538), (-71.421, -32.387), (-71.513, -32.208), (-71.526, -31.806), (-71.662, -31.17), (-71.654, -30.987), (-71.709, -30.628), (-71.669, -30.33), (-71.4, -30.143), (-71.316, -29.65), (-71.327, -29.443), (-71.353, -29.35), (-71.486, -29.198), (-71.519, -28.926), (-71.494, -28.855), (-71.384, -28.779), (-71.307, -28.672), (-71.267, -28.508), (-71.186, -28.378), (-71.154, -28.064), (-71.087, -27.814), (-71.053, -27.727), (-70.926, -27.589), (-70.898, -27.188), (-70.647, -26.329), (-70.635, -25.993), (-70.714, -25.784), (-70.633, -25.546), (-70.49, -25.376), (-70.452, -25.252), (-70.445, -25.173), (-70.574, -24.644), (-70.507, -24.13), (-70.507, -23.886), (-70.392, -23.566), (-70.512, -23.483), (-70.588, -23.368), (-70.593, -23.255), (-70.563, -23.057), (-70.45, -23.034), (-70.332, -22.849), (-70.088, -21.493), (-70.088, -21.253), (-70.197, -20.725), (-70.147, -20.23), (-70.157, -19.706), (-70.276, -19.268), (-70.362, -18.398), (-70.418, -18.346)],
  [(-73.773, -43.346), (-73.919, -43.372), (-74.114, -43.358), (-74.387, -43.232), (-74.209, -42.879), (-74.156, -42.591), (-74.199, -42.481), (-74.17, -42.269), (-74.072, -42.106), (-74.019, -41.891), (-74.063, -41.823), (-74.037, -41.796), (-73.731, -41.877), (-73.528, -41.896), (-73.454, -42.166), (-73.423, -42.193), (-73.439, -42.278), (-73.533, -42.314), (-73.525, -42.393), (-73.471, -42.466), (-73.789, -42.586), (-73.673, -42.704), (-73.568, -42.762), (-73.436, -42.937), (-73.541, -43.074), (-73.75, -43.159), (-73.738, -43.291), (-73.773, -43.346)],
]
print('outline rings:', len(CHILE_OUTLINE), '| vertices:', sum(len(r) for r in CHILE_OUTLINE))


In [ ]:
# === Load the dataset ======================================================
if TEST_MODE:
    rng = np.random.default_rng(42)
    n = 95000
    lat = rng.uniform(-42, -18, n)
    lon = -71 + rng.normal(0, 1.2, n) + 0.05 * (lat + 30)
    dtw = rng.exponential(11, n); neg = rng.random(n) < 0.05
    dtw[neg] = -rng.exponential(3, neg.sum())
    basin = np.array([f"Basin_{int((la + 18) // 2):02d}" for la in lat])
    df = pd.DataFrame({LON_COL: lon, LAT_COL: lat, DTW_COL: dtw, BASIN_COL: basin})
    print("TEST_MODE: synthetic dataset created.")
else:
    df = pd.read_excel(DATA_PATH) if DATA_PATH.lower().endswith((".xlsx", ".xls")) else pd.read_csv(DATA_PATH)
    print(f"Loaded {DATA_PATH}\nShape: {df.shape}")

required = [LON_COL, LAT_COL, DTW_COL, BASIN_COL]
missing = [c for c in required if c not in df.columns]
if missing:
    raise KeyError("Missing column(s): " + ", ".join(missing) +
                   "\n\nAvailable columns:\n  - " + "\n  - ".join(map(str, df.columns)))


In [ ]:
# === Clean (mirrors the manuscript) ========================================
n0 = len(df)
df = df.dropna(subset=[LON_COL, LAT_COL, DTW_COL]).copy()
n_neg = int((df[DTW_COL] < 0).sum())
df = df[df[DTW_COL] >= 0].copy()
print(f"Records: {n0:,} -> {len(df):,}  (removed {n_neg:,} negative-depth records)")
print(f"DTW (m): mean={df[DTW_COL].mean():.1f}, median={df[DTW_COL].median():.1f}, "
      f"min={df[DTW_COL].min():.2f}, max={df[DTW_COL].max():.1f}")
# NOTE: confirm this record count matches the value reported in the manuscript text.

agg = (df.groupby(BASIN_COL)
         .agg(lon=(LON_COL, "mean"), lat=(LAT_COL, "mean"),
              mean_dtw=(DTW_COL, "mean"), n=(DTW_COL, "size"))
         .reset_index())
print(f"Basins with data: {len(agg)}  |  records/basin: "
      f"min={agg['n'].min()}, median={int(agg['n'].median())}, max={agg['n'].max()}")


In [ ]:
# === Build Figure 3 ========================================================
plt.rcParams.update({"font.family": "serif", "font.size": 9,
                     "axes.titlesize": 10, "axes.linewidth": 0.6})
CMAP   = "viridis"
vmax   = float(np.percentile(df[DTW_COL], 95))
mlat   = float(df[LAT_COL].mean())
ASPECT = 1.0 / np.cos(np.radians(abs(mlat)))

# Latitude limits from the data; longitude limits widened to show the coastline.
lat_lim = (df[LAT_COL].min() - 0.6, df[LAT_COL].max() + 0.6)
_ox = [x for ring in CHILE_OUTLINE for (x, y) in ring if lat_lim[0] <= y <= lat_lim[1]]
lon_lo = min(df[LON_COL].min(), min(_ox)) - 0.35
lon_hi = max(df[LON_COL].max(), max(_ox)) + 0.35
lon_lim = (lon_lo, lon_hi)


def draw_outline(ax):
    for ring in CHILE_OUTLINE:
        xs = [p[0] for p in ring]; ys = [p[1] for p in ring]
        ax.plot(xs, ys, "-", color="0.35", lw=0.7, zorder=1)


def north_arrow(ax):
    ax.annotate("N", xy=(0.10, 0.965), xytext=(0.10, 0.875), xycoords="axes fraction",
                ha="center", va="center", fontsize=10, fontweight="bold",
                arrowprops=dict(arrowstyle="-|>", color="k", lw=1.3))


def scale_bar(ax, km=200):
    deg = km / (111.0 * np.cos(np.radians(abs(mlat))))
    x0 = lon_lim[0] + 0.08 * (lon_lim[1] - lon_lim[0])
    y0 = lat_lim[0] + 0.035 * (lat_lim[1] - lat_lim[0])
    ax.plot([x0, x0 + deg], [y0, y0], "k-", lw=2.4, solid_capstyle="butt", zorder=5)
    ax.text(x0 + deg / 2, y0 + 0.010 * (lat_lim[1] - lat_lim[0]), f"{km} km",
            ha="center", va="bottom", fontsize=7, zorder=5)


def setup_map(ax, title):
    draw_outline(ax)
    ax.set_aspect(ASPECT)
    ax.set_xlim(*lon_lim); ax.set_ylim(*lat_lim)
    ax.set_xlabel("Longitude (°)"); ax.set_ylabel("Latitude (°)")
    ax.set_title(title, fontweight="bold", loc="left")
    ax.grid(True, ls=":", lw=0.4, color="0.85")
    ax.set_facecolor("#eef3f9")


fig = plt.figure(figsize=(12.5, 8.6))
gs = GridSpec(2, 3, width_ratios=[1.0, 1.0, 1.05], height_ratios=[1, 1],
              wspace=0.34, hspace=0.36)
axA = fig.add_subplot(gs[:, 0]); axB = fig.add_subplot(gs[:, 1])
axC = fig.add_subplot(gs[0, 2]); axD = fig.add_subplot(gs[1, 2])

# ---- Panel A: basin aggregates ----
setup_map(axA, "(A) Basin-aggregated records")
axA.set_anchor("N")
# Bubble AREA is a compressed function of record count; modest max keeps bubbles readable.
SMIN, SMAX = 12, 300
nmin, nmax = agg["n"].min(), agg["n"].max()
def size_for(counts):
    frac = (np.asarray(counts, float) - nmin) / max(1, (nmax - nmin))
    return SMIN + np.sqrt(frac) * (SMAX - SMIN)   # sqrt compresses the large dynamic range
scA = axA.scatter(agg["lon"], agg["lat"], s=size_for(agg["n"]), c=agg["mean_dtw"],
                  cmap=CMAP, vmin=0, vmax=vmax, edgecolor="k", linewidth=0.4,
                  alpha=0.85, zorder=3)
north_arrow(axA)
cbA = fig.colorbar(scA, ax=axA, orientation="horizontal", fraction=0.045, pad=0.085)
cbA.set_label("Mean DTW per basin (m)", fontsize=8); cbA.ax.tick_params(labelsize=7)

# Clean size legend: nice round reference counts, solid white background, lower-left.
locs = MaxNLocator(nbins=4).tick_values(nmin, nmax)
ref = [int(v) for v in locs if nmin <= v <= nmax and v > 0][:3]
if len(ref) < 3:
    ref = sorted({int(nmin), int(np.median(agg["n"])), int(nmax)})
handles = [plt.scatter([], [], s=size_for(v), facecolor="0.7", edgecolor="k",
                       linewidth=0.4, label=f"{v:,}") for v in ref]
leg = axA.legend(handles=handles, title="Records / basin", loc="lower right",
                 frameon=True, framealpha=1.0, edgecolor="0.7", facecolor="white",
                 labelspacing=1.5, borderpad=0.9, handletextpad=1.3,
                 fontsize=7.5, title_fontsize=7.5)
leg.set_zorder(6)

# ---- Panel B: individual records ----
setup_map(axB, "(B) Individual records")
axB.set_anchor("N")
scB = axB.scatter(df[LON_COL], df[LAT_COL], c=df[DTW_COL], cmap=CMAP, vmin=0, vmax=vmax,
                  s=4, alpha=0.45, edgecolor="none", rasterized=True, zorder=3)
north_arrow(axB); scale_bar(axB)
axB.text(0.96, 0.965, f"n = {len(df):,}", transform=axB.transAxes, ha="right", va="top",
         fontsize=8, bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="0.7", lw=0.6), zorder=6)
cbB = fig.colorbar(scB, ax=axB, orientation="horizontal", fraction=0.045, pad=0.085)
cbB.set_label("Depth to water (m)", fontsize=8); cbB.ax.tick_params(labelsize=7)

# ---- Panel C: DTW distribution ----
axC.hist(df[DTW_COL], bins=60, range=(0, float(df[DTW_COL].max())),
         color="#4C72B0", edgecolor="white", linewidth=0.3)
axC.axvline(df[DTW_COL].mean(),   color="#C44E52", ls="--", lw=1.2, label=f"mean = {df[DTW_COL].mean():.1f} m")
axC.axvline(df[DTW_COL].median(), color="k",       ls=":",  lw=1.2, label=f"median = {df[DTW_COL].median():.1f} m")
axC.set_title("(C) DTW distribution", fontweight="bold", loc="left")
axC.set_xlabel("Depth to water (m)"); axC.set_ylabel("Frequency")
axC.legend(fontsize=7.5, frameon=False)

# ---- Panel D: log-transformed distribution ----
axD.hist(np.log1p(df[DTW_COL]), bins=60, color="#DD8452", edgecolor="white", linewidth=0.3)
axD.set_title("(D) Log-transformed distribution", fontweight="bold", loc="left")
axD.set_xlabel("log(1 + DTW)"); axD.set_ylabel("Frequency")

fig.suptitle("Figure 3. Spatial distribution of groundwater data and the target variable (Depth to Water)",
             fontsize=11, fontweight="bold", y=0.99)
plt.show()


In [ ]:
# === Save (PNG 400 dpi + PDF + SVG) ========================================
stem = os.path.join(OUTPUT_DIR, "Figure3_DTW_spatial_distribution")
fig.savefig(stem + ".png", dpi=400, bbox_inches="tight")
fig.savefig(stem + ".pdf", bbox_inches="tight")
fig.savefig(stem + ".svg", bbox_inches="tight")
print("Saved:")
for ext in (".png", ".pdf", ".svg"):
    print("  ", stem + ext)
